In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F


#create Spark Session
spark = SparkSession.builder.appName("water_quality_agg_gold").getOrCreate()

###Load Silver Layer

In [0]:
silver_df=spark.read.format("delta").load("/mnt/adls/silver/water_quality_enriched")


In [0]:
#drop Duplicates
gold_df=silver_df.dropDuplicates()


In [0]:
# Calculate mean and standard deviation for 'minimum_value' column
mean_val = gold_df.select(F.mean("Minimum_Value")).first()[0]
stddev_val = gold_df.select(F.stddev("Minimum_Value")).first()[0]


# Calculate Z-Score and identify outliers
gold_df_with_zscore = gold_df.withColumn(
    "z_score", (F.col("Minimum_Value") - mean_val) / stddev_val
)
gold_df_with_outliers = gold_df_with_zscore.withColumn(
    "MinimumValue_outlier", F.when(
        F.abs(F.col("z_score")) > 3, 1
    ).otherwise(0)
)



In [0]:
gold_df = gold_df_with_outliers

# Calculate mean and standard deviation for 'Max_Value' column
mean_val = gold_df.select(F.mean("Maximum_Value")).first()[0]
stddev_val = gold_df.select(F.stddev("Maximum_Value")).first()[0]


# Calculate Z-Score and identify outliers
gold_df_with_zscore = gold_df.withColumn(
    "z_score", (F.col("Maximum_Value") - mean_val) / stddev_val
)
gold_df_with_outliers = gold_df_with_zscore.withColumn(
    "MaxValue_outlier", F.when(
        F.abs(F.col("z_score")) > 3, 1
    ).otherwise(0)
)




In [0]:
gold_df = gold_df_with_outliers


In [0]:
# Calculate mean and standard deviation for 'Max_Value' column
mean_val = gold_df.select(F.mean("Mean_Value")).first()[0]
stddev_val = gold_df.select(F.stddev("Mean_Value")).first()[0]


# Calculate Z-Score and identify outliers
gold_df_with_zscore = gold_df.withColumn(
    "z_score", (F.col("Mean_Value") - mean_val) / stddev_val
)
gold_df_with_outliers = gold_df_with_zscore.withColumn(
    "MeanValue_outlier", F.when(
        F.abs(F.col("z_score")) > 3, 1
    ).otherwise(0)
)


# Show results

In [0]:
# Step 2: Filter out rows where MinimumValue_outlier is 1
gold_df_no_outliers = gold_df_with_outliers.filter(gold_df_with_outliers.MeanValue_outlier == 0)

gold_df = gold_df_no_outliers
gold_df = gold_df.drop("z_score")


In [0]:
# Handle sampling period by splitting into start and end dates
gold_df = gold_df.withColumn("Start_Date", col("Sampling_Period").substr(1, 10)) \
    .withColumn("End_Date", col("Sampling_Period").substr(-10, 10))

In [0]:
# Write the final data to the Gold layer
gold_df.write.format("delta").mode("overwrite").save("/mnt/adls/gold/water_quality_aggregated")

In [0]:
#We have to create a database under the hive metastore which is present in the workspace inside the catalohg option on the left hand side of the screen
spark.sql("CREATE DATABASE IF NOT EXISTS waterdb")

In [0]:
gold_df.write.format("csv").mode("overwrite").saveAsTable("waterdb.gold_table")